# 010_decoding_compression_ratio_condition_specific.ipynb

Compression-ratio comparison notebook for the condition-specific ranking branch. Loads the new `msaa_condrank_decoding_outputs_*` directories.


In [ ]:
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

FIT_SCOPE = "across"  # "within" or "across"

DECODE_DIR_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_{fit_scope}"

ANALYSIS_TYPES = ["spatial", "temporal"]
CONDITIONS = ["intact", "word", "rest"]

COND_COLORS = {
    "intact": "purple",
    "word": "green",
    "rest": "black",
}

ANALYSIS_LINESTYLES = {
    "spatial": "--",
    "temporal": "-",
}

# Dimensions of your data
T = 300   # number of timepoints
V = 700   # number of nodes/features

# Number of subjects per condition.
# If None, the notebook will try to infer N from count columns, but that count is usually decoding repetitions,
# not subject count. So manually setting N is recommended for parameter-count analysis.
N_SUBJECTS_PER_CONDITION = None
# Example:
# N_SUBJECTS_PER_CONDITION = 36

# Plot choices
USE_TOPM = False
TOP_M_TO_COMPARE = None
# Example:
# USE_TOPM = True
# TOP_M_TO_COMPARE = 5

K_VALUES_TO_PLOT = None
# Example:
K_VALUES_TO_PLOT = [10, 25, 50, 75, 100]

FIGURE_OUT_DIR = "010_paper_compression_ratio_figures_condition_specific"
SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300

FIGSIZE = (8, 5)
FONT_SIZE = 12
TITLE_SIZE = 13
LABEL_SIZE = 12
TICK_SIZE = 10
LEGEND_SIZE = 10
LINEWIDTH = 2.2
MARKER_SIZE = 6
CAPSIZE = 4

YLIM = None

In [ ]:

# ============================================================
# IMPORTS
# ============================================================

%matplotlib inline

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path(FIGURE_OUT_DIR).mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "axes.labelsize": LABEL_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
    "legend.fontsize": LEGEND_SIZE,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Ready.")

In [ ]:
# ============================================================
# FIGURE SAVING SETTINGS -- EXPLICIT, NO RECURSION
# ============================================================

from pathlib import Path

SAVE_FIGS = True
FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "010_decoding_compression_ratio"
FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_FORMAT = "pdf"   # "pdf", "png", or "svg"
DPI = 300

FIG_DIR.mkdir(parents=True, exist_ok=True)

_fig_counter = 0

def _sanitize_fig_name(name):
    name = str(name).replace(" ", "_").replace("|", "_").replace("/", "-").replace("\\", "-")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:160] if name else "figure"

def _figure_has_content(fig=None):
    if fig is None:
        fig = plt.gcf()
    if len(fig.axes) == 0:
        return False
    for ax in fig.axes:
        if ax.lines or ax.collections or ax.images or ax.patches or ax.texts or ax.get_title():
            return True
    return True

def save_current_fig(name=None):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    fig = plt.gcf()
    if not _figure_has_content(fig):
        return None
    _fig_counter += 1
    if name is None:
        try:
            title = plt.gca().get_title()
        except Exception:
            title = ""
        label = _sanitize_fig_name(title if title else "figure")
    else:
        label = _sanitize_fig_name(name)
    out = FIG_DIR / f"{_fig_counter:03d}_{label}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def savefig(name=None, force=True):
    return save_current_fig(name=name)

print("Figure directory:", FIG_DIR)
print("Explicit save mode: plt.show is not patched.")


In [ ]:
# ============================================================
# CACHE SETTINGS
# ============================================================

from pathlib import Path
import pickle

CACHE_DIR = Path("010_decoding_compression_ratio_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

USE_CACHE = True
OVERWRITE_CACHE = False

CLUSTER_SUMMARY_CACHE = CACHE_DIR / "cluster_summary_df.csv"
SELECTED_ARCHETYPES_CACHE = CACHE_DIR / "selected_archetypes_dict.npy"
PLOT_DATA_CACHE = CACHE_DIR / "plot_data_cache.pkl"

print("Cache directory:", CACHE_DIR.resolve())
print("USE_CACHE:", USE_CACHE)
print("OVERWRITE_CACHE:", OVERWRITE_CACHE)

## Load saved decoding summaries

In [ ]:

def load_csv(path):
    path = Path(path)
    if path.exists():
        print("Loaded:", path)
        return pd.read_csv(path)
    print("Missing:", path)
    return None


def standardize_decoding_df(df, analysis_type, fit_scope):
    if df is None:
        return None

    df = df.copy()

    rename_map = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename_map["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "err" not in df.columns:
        rename_map["sem_accuracy"] = "err"
    if "std_accuracy" in df.columns and "std" not in df.columns:
        rename_map["std_accuracy"] = "std"
    if "sem" in df.columns and "err" not in df.columns:
        rename_map["sem"] = "err"

    df = df.rename(columns=rename_map)

    if "analysis_type" not in df.columns:
        df["analysis_type"] = analysis_type
    if "fit_scope" not in df.columns:
        df["fit_scope"] = fit_scope

    df["analysis_type"] = df["analysis_type"].astype(str)
    df["fit_scope"] = df["fit_scope"].astype(str)

    if "K" in df.columns:
        df["K"] = df["K"].astype(int)
    if "top_m" in df.columns:
        df["top_m"] = df["top_m"].astype(int)
    if "condition" in df.columns:
        df["condition"] = df["condition"].astype(str)

    return df


def load_decoding_summary(analysis_type, fit_scope, use_topm=False):
    d = Path(DECODE_DIR_TEMPLATE.format(analysis_type=analysis_type, fit_scope=fit_scope))

    if use_topm:
        path = d / "topm_summary.csv"
    else:
        path = d / "full_summary.csv"

    df = load_csv(path)
    df = standardize_decoding_df(df, analysis_type, fit_scope)

    if df is None:
        return None

    df = df[
        (df["analysis_type"] == analysis_type) &
        (df["fit_scope"] == fit_scope)
    ].copy()

    if K_VALUES_TO_PLOT is not None:
        df = df[df["K"].isin(K_VALUES_TO_PLOT)].copy()

    if use_topm:
        if "top_m" not in df.columns:
            raise ValueError(f"Requested top-m comparison but no top_m column found in {path}")
        if TOP_M_TO_COMPARE is None:
            available = sorted(df["top_m"].unique())
            raise ValueError(f"Set TOP_M_TO_COMPARE. Available top_m values: {available}")
        df = df[df["top_m"] == TOP_M_TO_COMPARE].copy()

    return df


dfs = []
for analysis_type in ANALYSIS_TYPES:
    df = load_decoding_summary(analysis_type, FIT_SCOPE, use_topm=USE_TOPM)
    if df is not None:
        dfs.append(df)

decode_df = pd.concat(dfs, ignore_index=True)
display(decode_df.head())
print("Loaded rows:", len(decode_df))
print("Columns:", list(decode_df.columns))


## Compute compression metrics

We compute several normalization measures.

### 1. Axis-specific compression ratio

For spatial AA, the shared archetype axis is time:

\[
\mathrm{compression}_{spatial} = K / T
\]

For temporal AA, the shared archetype axis is nodes:

\[
\mathrm{compression}_{temporal} = K / V
\]

This asks: what fraction of the decomposed axis is represented by `K` archetypes?

### 2. Approximate multisubject parameter count

For spatial AA:

\[
K T + N K V
\]

For temporal AA:

\[
K V + N K T
\]

This distinguishes shared parameters from subject-specific parameters.

### 3. Normalized parameter fraction

We divide approximate parameters by the full data size:

\[
N T V
\]

This is not perfect, but it is useful for comparing the rough representational capacity of the two orientations.


In [ ]:

def add_compression_metrics(df, T=300, V=700, n_subjects=None):
    df = df.copy()

    if n_subjects is None:
        # Fallback. This is not ideal, but keeps the notebook runnable.
        # For parameter-count claims, manually set N_SUBJECTS_PER_CONDITION.
        n_subjects = np.nan

    df["axis_dim"] = np.where(df["analysis_type"] == "spatial", T, V)
    df["axis_compression_ratio"] = df["K"] / df["axis_dim"]

    df["shared_params"] = np.where(
        df["analysis_type"] == "spatial",
        df["K"] * T,
        df["K"] * V
    )

    if np.isnan(n_subjects):
        df["subject_specific_params"] = np.nan
        df["approx_total_params"] = np.nan
        df["param_fraction"] = np.nan
    else:
        df["subject_specific_params"] = np.where(
            df["analysis_type"] == "spatial",
            n_subjects * df["K"] * V,
            n_subjects * df["K"] * T
        )
        df["approx_total_params"] = df["shared_params"] + df["subject_specific_params"]
        df["param_fraction"] = df["approx_total_params"] / (n_subjects * T * V)

    return df


plot_df = add_compression_metrics(
    decode_df,
    T=T,
    V=V,
    n_subjects=N_SUBJECTS_PER_CONDITION
)

display(plot_df.head())

## Helper plotting functions

In [ ]:

def get_err_col(df):
    for col in ["err", "sem", "sem_accuracy", "stderr", "se"]:
        if col in df.columns:
            return col
    return None


def savefig(name):
    if not SAVE_FIGS:
        return
    suffix = f"topm{TOP_M_TO_COMPARE}" if USE_TOPM else "full"
    out = Path(FIGURE_OUT_DIR) / f"{name}_{FIT_SCOPE}_{suffix}.{FIG_FORMAT}"
    plt.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)


def plot_metric_x(df, x_col, xlabel, title, fname):
    err_col = get_err_col(df)

    plt.figure(figsize=FIGSIZE)

    for analysis_type in ANALYSIS_TYPES:
        for cond in CONDITIONS:
            sub = df[
                (df["analysis_type"] == analysis_type) &
                (df["condition"] == cond)
            ].sort_values(x_col)

            if len(sub) == 0:
                continue

            yerr = sub[err_col] if err_col is not None else None

            plt.errorbar(
                sub[x_col],
                sub["mean"],
                yerr=yerr,
                color=COND_COLORS[cond],
                linestyle=ANALYSIS_LINESTYLES[analysis_type],
                marker="o",
                markersize=MARKER_SIZE,
                linewidth=LINEWIDTH,
                capsize=CAPSIZE,
                label=f"{analysis_type} | {cond}",
            )

    if YLIM is not None:
        plt.ylim(*YLIM)

    plt.xlabel(xlabel)
    plt.ylabel("Decoding accuracy")
    plt.title(title)
    plt.legend(frameon=False, ncol=2)
    plt.tight_layout()
    savefig(fname)
    save_current_fig()
    plt.show()
    plt.close()

# Plot 1: Decoding versus raw K

In [ ]:

plot_metric_x(
    plot_df,
    x_col="K",
    xlabel="Number of archetypes (K)",
    title=f"Decoding vs raw K | {FIT_SCOPE}",
    fname="decoding_vs_rawK"
)

# Plot 2: Decoding versus axis-specific compression ratio

In [ ]:

plot_metric_x(
    plot_df,
    x_col="axis_compression_ratio",
    xlabel="Axis-specific compression ratio (K / decomposed axis dimension)",
    title=f"Decoding vs axis-specific compression ratio | {FIT_SCOPE}",
    fname="decoding_vs_axis_compression_ratio"
)


# Plot 3: Decoding versus approximate parameter fraction

This plot requires `N_SUBJECTS_PER_CONDITION` to be set.

If it is `None`, this cell will print a warning and skip the plot.


In [ ]:

if N_SUBJECTS_PER_CONDITION is None:
    print("Skipping parameter-fraction plot. Set N_SUBJECTS_PER_CONDITION to use this.")
else:
    plot_metric_x(
        plot_df,
        x_col="param_fraction",
        xlabel="Approximate parameter fraction",
        title=f"Decoding vs approximate parameter fraction | {FIT_SCOPE}",
        fname="decoding_vs_param_fraction"
    )


# Plot 4: Matched compression comparison table

This table helps you see which spatial and temporal `K` values are closest in compression ratio.

For example, temporal `K=50` has compression ratio `50/700`, which is closest to spatial `K≈21` for `T=300`.


In [ ]:

def make_matched_compression_table(K_values, T=300, V=700):
    rows = []
    K_values = sorted(K_values)

    for K_temporal in K_values:
        target_ratio = K_temporal / V
        matched_spatial = min(K_values, key=lambda k: abs((k / T) - target_ratio))

        rows.append({
            "temporal_K": K_temporal,
            "temporal_ratio": K_temporal / V,
            "nearest_spatial_K": matched_spatial,
            "spatial_ratio": matched_spatial / T,
            "ratio_difference": abs((matched_spatial / T) - target_ratio),
        })

    return pd.DataFrame(rows)


available_K = sorted(plot_df["K"].unique())
matched_table = make_matched_compression_table(available_K, T=T, V=V)
display(matched_table)


# Plot 5: Best decoding at or below matched compression thresholds

This summarizes, for each condition and analysis type, the best decoding accuracy available up to a given compression ratio.

This can help support a more careful claim such as:

> Temporal AA reaches a given decoding accuracy at lower or comparable compression levels.


In [ ]:

thresholds = np.linspace(
    plot_df["axis_compression_ratio"].min(),
    plot_df["axis_compression_ratio"].max(),
    25
)

auc_rows = []

for analysis_type in ANALYSIS_TYPES:
    for cond in CONDITIONS:
        sub = plot_df[
            (plot_df["analysis_type"] == analysis_type) &
            (plot_df["condition"] == cond)
        ].sort_values("axis_compression_ratio")

        if len(sub) == 0:
            continue

        for thr in thresholds:
            eligible = sub[sub["axis_compression_ratio"] <= thr]
            if len(eligible) == 0:
                best = np.nan
            else:
                best = eligible["mean"].max()

            auc_rows.append({
                "analysis_type": analysis_type,
                "condition": cond,
                "compression_threshold": thr,
                "best_decoding_up_to_threshold": best,
            })

threshold_df = pd.DataFrame(auc_rows)

plt.figure(figsize=FIGSIZE)

for analysis_type in ANALYSIS_TYPES:
    for cond in CONDITIONS:
        sub = threshold_df[
            (threshold_df["analysis_type"] == analysis_type) &
            (threshold_df["condition"] == cond)
        ]
        if len(sub) == 0:
            continue

        plt.plot(
            sub["compression_threshold"],
            sub["best_decoding_up_to_threshold"],
            color=COND_COLORS[cond],
            linestyle=ANALYSIS_LINESTYLES[analysis_type],
            linewidth=LINEWIDTH,
            label=f"{analysis_type} | {cond}",
        )

if YLIM is not None:
    plt.ylim(*YLIM)

plt.xlabel("Compression ratio threshold")
plt.ylabel("Best decoding up to threshold")
plt.title(f"Best decoding achievable by compression threshold | {FIT_SCOPE}")
plt.legend(frameon=False, ncol=2)
plt.tight_layout()
savefig("best_decoding_by_compression_threshold")
save_current_fig()
plt.show()
plt.close()
display(threshold_df.head())


# Save comparison table

This saves the long-format dataframe with compression metrics added.


In [ ]:

out_csv = Path(FIGURE_OUT_DIR) / f"decoding_with_compression_metrics_{FIT_SCOPE}.csv"
plot_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)